
# Step 2 - Auditoria do caso duplicado SISBRA -> FDSN

Objetivo: mostrar, de forma linear, o caso em que duas linhas do SISBRA
convergiram para um unico `fdsn.resource_id`, como o pipeline escolheu o
evento canonico e como a consolidacao ficou registrada no `event.json` final.

Entradas principais:
- `outputs/events_duplicate_merge_report.csv`
- `outputs/events_materialize_report.csv`
- `data/events/2021078T053957/event.json`
- `data/events_stage/20210319T053957_usp2021flcy_row744/event.json`

Este notebook é self-contained: não importa scripts Python do repositório e não executa bash.


In [1]:

from pathlib import Path
import json

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)


In [2]:

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / ".git").exists():
    repo_root = repo_root.parent

if not (repo_root / ".git").exists():
    raise RuntimeError("Nao foi possivel localizar a raiz do repositorio a partir do diretorio atual.")

duplicate_report_csv = repo_root / "outputs/events_duplicate_merge_report.csv"
materialize_report_csv = repo_root / "outputs/events_materialize_report.csv"
final_event_json = repo_root / "data/events/2021078T053957/event.json"
duplicate_stage_event_json = repo_root / "data/events_stage/20210319T053957_usp2021flcy_row744/event.json"
tables_dir = repo_root / "outputs/tables"
tables_dir.mkdir(parents=True, exist_ok=True)

for path in [duplicate_report_csv, materialize_report_csv, final_event_json, duplicate_stage_event_json]:
    print(path.relative_to(repo_root), "->", path.exists())


outputs/events_duplicate_merge_report.csv -> True
outputs/events_materialize_report.csv -> True
data/events/2021078T053957/event.json -> True
data/events_stage/20210319T053957_usp2021flcy_row744/event.json -> True


In [3]:

duplicate_report = pd.read_csv(duplicate_report_csv)
materialize_report = pd.read_csv(materialize_report_csv)

print("Grupos de merge detectados:", len(duplicate_report))
display(duplicate_report)


Grupos de merge detectados: 1


,merge_key,merge_key_type,target_folder,canonical_source_folder,canonical_rownum,canonical_rownum_source,canonical_match_dt_s,canonical_match_dist_km,duplicate_source_folder,duplicate_rownum,duplicate_rownum_source,duplicate_match_dt_s,duplicate_match_dist_km,group_size,merge_status,notes
0,smi:org.gfz-potsdam.de/geofon/usp2021flcy,fdsn.resource_id,2021078T053957,20210319T053957_usp2021flcy_row743,743,4875,0.113955,2.147325,20210319T053957_usp2021flcy_row744,744,4878,0.886045,4.709782,2,merged_duplicate,same_fdsn_resource_id



## 1. Leitura do grupo de merge

No rerun M1.2 apareceu um unico grupo de duplicacao. O relatorio abaixo ja
mostra o essencial: mesma chave FDSN, mesmo diretorio final e duas linhas
SISBRA concorrendo entre si.


In [4]:

merge_row = duplicate_report.iloc[0].copy()

merge_overview = pd.DataFrame([
    {
        "papel": "canonico",
        "source_folder": merge_row["canonical_source_folder"],
        "rownum": merge_row["canonical_rownum"],
        "rownum_source": merge_row["canonical_rownum_source"],
        "match_dt_s": merge_row["canonical_match_dt_s"],
        "match_dist_km": merge_row["canonical_match_dist_km"],
    },
    {
        "papel": "duplicado absorvido",
        "source_folder": merge_row["duplicate_source_folder"],
        "rownum": merge_row["duplicate_rownum"],
        "rownum_source": merge_row["duplicate_rownum_source"],
        "match_dt_s": merge_row["duplicate_match_dt_s"],
        "match_dist_km": merge_row["duplicate_match_dist_km"],
    },
])

display(merge_overview)

print("merge_key:", merge_row["merge_key"])
print("target_folder final:", merge_row["target_folder"])
print("group_size:", merge_row["group_size"])

merge_overview.to_csv(tables_dir / "step2_duplicate_case_overview.csv", index=False)


,papel,source_folder,rownum,rownum_source,match_dt_s,match_dist_km
0,canonico,20210319T053957_usp2021flcy_row743,743,4875,0.113955,2.147325
1,duplicado absorvido,20210319T053957_usp2021flcy_row744,744,4878,0.886045,4.709782


merge_key: smi:org.gfz-potsdam.de/geofon/usp2021flcy
target_folder final: 2021078T053957
group_size: 2



## 2. Comparacao das linhas SISBRA

A linha canonica vem do `event.json` final. A linha absorvida continua
preservada em `sisbra_duplicates` e tambem pode ser conferida no bundle de
stage que nao foi materializado no dataset final.


In [5]:

final_event = json.loads(final_event_json.read_text())
duplicate_stage_event = json.loads(duplicate_stage_event_json.read_text())

canonical_sisbra = final_event["sisbra"]
duplicate_sisbra = final_event["sisbra_duplicates"][0]
fdsn_event = final_event["fdsn"]

sisbra_side_by_side = pd.DataFrame([
    {"papel": "canonico", **canonical_sisbra},
    {"papel": "duplicado absorvido", **duplicate_sisbra},
])[[
    "papel",
    "rownum",
    "rownum_source",
    "origin_time",
    "latitude",
    "longitude",
    "magnitude",
    "depth_km",
    "state",
    "localities",
    "source_comments",
]]

display(sisbra_side_by_side)

print(
    "Linha absorvida confere com o bundle de stage:",
    duplicate_stage_event["sisbra"]["rownum_source"] == duplicate_sisbra["rownum_source"],
)

sisbra_side_by_side.to_csv(tables_dir / "step2_duplicate_case_sisbra_side_by_side.csv", index=False)


,papel,rownum,rownum_source,origin_time,latitude,longitude,magnitude,depth_km,state,localities,source_comments
0,canonico,743,4875,2021-03-19T05:39:58Z,-20.47,-43.95,2.1,0.0,MG,Jeceaba,(USP; UnB)
1,duplicado absorvido,744,4878,2021-03-19T05:39:57Z,-20.43,-43.99,2.0,0.0,MG,Belo_Vale_Congonhas,(RSVL-USP)


Linha absorvida confere com o bundle de stage: True



## 3. Criterio de canonicalizacao

A politica operacional do projeto e: um unico evento final por
`fdsn.resource_id`. Quando ha mais de uma linha SISBRA no mesmo grupo, o
canonico e escolhido por ordem deterministica: `dt_s`, `dist_km`,
`waveforms_ok_count`, `sisbra_rownum` e `source_folder`.


In [6]:
D
target_folder = merge_row["target_folder"]
canonical_folder = merge_row["canonical_source_folder"]
duplicate_folder = merge_row["duplicate_source_folder"]

ranking_rows = materialize_report[
    materialize_report["source_folder"].isin([canonical_folder, duplicate_folder])
].copy()
ranking_view = ranking_rows[[
    "source_folder",
    "action",
    "match_dt_s",
    "match_dist_km",
    "waveforms_ok_count",
    "sisbra_rownum",
    "sisbra_rownum_source",
    "fdsn_resource_id",
]].copy()
ranking_view = ranking_view.sort_values(
    ["match_dt_s", "match_dist_km", "waveforms_ok_count", "sisbra_rownum", "source_folder"],
    ascending=[True, True, False, True, True],
).reset_index(drop=True)

display(ranking_view)

ranking_view.to_csv(tables_dir / "step2_duplicate_case_ranking.csv", index=False)


,source_folder,action,match_dt_s,match_dist_km,waveforms_ok_count,sisbra_rownum,sisbra_rownum_source,fdsn_resource_id
0,20210319T053957_usp2021flcy_row743,moved,0.113955,2.147325,2,743,4875,smi:org.gfz-potsdam.de/geofon/usp2021flcy
1,20210319T053957_usp2021flcy_row744,merged_duplicate,0.886045,4.709782,2,744,4878,smi:org.gfz-potsdam.de/geofon/usp2021flcy



## 4. Evento final consolidado

O `event.json` final nao perde informacao. A entrada SISBRA canonica
continua no bloco `sisbra`, enquanto as demais linhas do mesmo grupo ficam
registradas em `sisbra_duplicates` e no bloco `dedup`.


In [7]:
dedup_view = pd.DataFrame([final_event["dedup"]])
duplicates_view = pd.DataFrame(final_event["sisbra_duplicates"])
fdsn_view = pd.DataFrame([fdsn_event])[[
    "resource_id",
    "origin_time",
    "latitude",
    "longitude",
    "magnitude",
    "depth_m",
    "agency_id",
]]
fdsn_view = fdsn_view.rename(columns={"depth_m": "depth_m_fdsn"})

print("Evento FDSN consolidado:")
display(fdsn_view)

print("Bloco dedup salvo no event.json final:")
display(dedup_view)

print("Linhas SISBRA absorvidas:")
display(duplicates_view)

dedup_view.to_csv(tables_dir / "step2_duplicate_case_dedup_block.csv", index=False)


Evento FDSN consolidado:


,resource_id,origin_time,latitude,longitude,magnitude,depth_m_fdsn,agency_id
0,smi:org.gfz-potsdam.de/geofon/usp2021flcy,2021-03-19T05:39:57.886045Z,-20.46837,-43.970509,2.20551,0.0,USP


Bloco dedup salvo no event.json final:


,canonical_sisbra_rownum,canonical_sisbra_rownum_source,canonical_source_folder,merge_key,merge_key_type,merged_count,merged_sisbra_rownums,merged_sisbra_rownums_source,merged_source_folders,policy
0,743,4875,20210319T053957_usp2021flcy_row743,smi:org.gfz-potsdam.de/geofon/usp2021flcy,fdsn.resource_id,2,"[743, 744]","[4875, 4878]","[20210319T053957_usp2021flcy_row743, 20210319T053957_usp2021flcy_row744]",merge_by_fdsn


Linhas SISBRA absorvidas:


,depth_km,latitude,localities,longitude,magnitude,match_dist_km,match_dt_s,origin_time,rownum,rownum_source,source_comments,source_folder,state
0,0.0,-20.43,Belo_Vale_Congonhas,-43.99,2.0,4.709782,0.886045,2021-03-19T05:39:57Z,744,4878,(RSVL-USP),20210319T053957_usp2021flcy_row744,MG



## 5. Conclusao

- O problema nao era um falso alarme: duas linhas SISBRA diferentes realmente casaram com o mesmo evento FDSN.
- O pipeline agora resolve isso sem perder rastreabilidade: um evento final por `fdsn.resource_id`.
- O caso `usp2021flcy` fica pronto para discussao com os professores porque a duplicidade, o criterio de escolha e a linha absorvida ficaram explicitados.
